In [1]:
%pip install scikit-learn pandas sentence-transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 780.3/780.3 kB 10.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 857.0/857.0 kB 15.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 15.6 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.6/13.6 MB 19.0 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 16.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 138.5/138.5 MB 26.0 MB/s  0:00:05m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.3/6.3 MB 22.9 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 536.2/536.2 kB 14.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19/19 [sentence-transformers]ence-transformers]

[notice] A new release of pip is available: 26.2.1 -> 26.3
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# Intentamos cargar Sentence-Transformers para similitud semántica real (embeddings).
# Si no está instalado o hay límite de memoria en el despliegue, usa TF-IDF automáticamente.
try:
    from sentence_transformers import SentenceTransformer
    EMBEDDINGS_DISPONIBLES = True
except ImportError:
    EMBEDDINGS_DISPONIBLES = False

/Users/jairmsmz/.pyenv/versions/3.11.11/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
W1124 18:44:08.025000 42802 site-packages/torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.


# Base de conocimiento de Nexia Analytics & Software

In [3]:
def cargar_base_conocimiento() -> pd.DataFrame:
    datos = [
        # --- Políticas ---
        {"id": "DOC-01", "categoria": "Política", "actividad": "Diseño_Arquitectura_Software", "tecnologia": "GitHub",
         "texto": "Guía de estilo y buenas prácticas para escribir código limpio (Clean Code) en Python."},
        {"id": "DOC-02", "categoria": "Política", "actividad": "Implementación_Machine_Learning", "tecnologia": "Base_Conocimiento",
         "texto": "Lineamientos de gobernanza de datos y privacidad de la información del cliente."},
        {"id": "DOC-03", "categoria": "Política", "actividad": "Diseño_Arquitectura_Software", "tecnologia": "GitHub",
         "texto": "Reglamento interno y políticas de la empresa para el uso de GitHub y repositorios."},
        {"id": "DOC-04", "categoria": "Política", "actividad": "Resolución_Incidentes_Críticos", "tecnologia": "Entorno_Cloud",
         "texto": "Políticas de seguridad de la información, manejo de contraseñas y accesos VPN."},
        {"id": "DOC-05", "categoria": "Política", "actividad": "Diseño_Arquitectura_Software", "tecnologia": "Base_Conocimiento",
         "texto": "Reglamento de confidencialidad y acuerdos de no divulgación (NDA) para consultores."},
        {"id": "DOC-06", "categoria": "Política", "actividad": "Diseño_Arquitectura_Software", "tecnologia": "GitHub",
         "texto": "Lineamientos estándar para la revisión de código (Pull Requests) entre pares."},
        {"id": "DOC-07", "categoria": "Política", "actividad": "Resolución_Incidentes_Críticos", "tecnologia": "Base_Conocimiento",
         "texto": "Normativa oficial de la empresa sobre horarios laborales y uso de equipos físicos."},
        {"id": "DOC-08", "categoria": "Política", "actividad": "Resolución_Incidentes_Críticos", "tecnologia": "Base_Conocimiento",
         "texto": "Políticas de uso aceptable de dispositivos personales y corporativos."},

        # --- Técnicos ---
        {"id": "DOC-09", "categoria": "Técnico", "actividad": "Diseño_Arquitectura_Software", "tecnologia": "Entorno_Cloud",
         "texto": "Manual de arquitectura de software para el nuevo cliente corporativo."},
        {"id": "DOC-10", "categoria": "Técnico", "actividad": "Implementación_Machine_Learning", "tecnologia": "Base_Conocimiento",
         "texto": "Diccionario de datos para el pipeline del modelo de Machine Learning."},
        {"id": "DOC-11", "categoria": "Técnico", "actividad": "Diseño_Arquitectura_Software", "tecnologia": "GitHub",
         "texto": "Documentación técnica de la API REST para el manejo de usuarios."},
        {"id": "DOC-12", "categoria": "Técnico", "actividad": "Diseño_Arquitectura_Software", "tecnologia": "Entorno_Cloud",
         "texto": "Diagramas de flujo de datos y arquitectura de microservicios en AWS."},
        {"id": "DOC-13", "categoria": "Técnico", "actividad": "Diseño_Arquitectura_Software", "tecnologia": "Entorno_Cloud",
         "texto": "Esquema relacional de la base de datos PostgreSQL para el módulo de facturación."},
        {"id": "DOC-14", "categoria": "Técnico", "actividad": "Diseño_Arquitectura_Software", "tecnologia": "Entorno_Cloud",
         "texto": "Especificaciones de los endpoints de integración con la pasarela de pagos."},

        # --- Operativos ---
        {"id": "DOC-15", "categoria": "Operativo", "actividad": "Resolución_Incidentes_Críticos", "tecnologia": "Entorno_Cloud",
         "texto": "Reporte post-mortem del fallo en el clúster de producción de AWS."},
        {"id": "DOC-16", "categoria": "Operativo", "actividad": "Diseño_Arquitectura_Software", "tecnologia": "Contenedor",
         "texto": "Guía de despliegue de contenedores utilizando Docker y Kubernetes."},
        {"id": "DOC-17", "categoria": "Operativo", "actividad": "Resolución_Incidentes_Críticos", "tecnologia": "Entorno_Cloud",
         "texto": "Ticket de resolución de incidente crítico de latencia en la base de datos."},
        {"id": "DOC-18", "categoria": "Operativo", "actividad": "Diseño_Arquitectura_Software", "tecnologia": "Base_Conocimiento",
         "texto": "Manual de usuario paso a paso para el sistema interno de gestión de viáticos."},
        {"id": "DOC-19", "categoria": "Operativo", "actividad": "Resolución_Incidentes_Críticos", "tecnologia": "Entorno_Cloud",
         "texto": "Bitácora de mantenimiento semanal y rotación de credenciales en servidores de desarrollo."},
        {"id": "DOC-20", "categoria": "Operativo", "actividad": "Resolución_Incidentes_Críticos", "tecnologia": "Base_Conocimiento",
         "texto": "Reporte mensual de métricas de incidencias (SLA) y tiempos de respuesta de soporte."},
        {"id": "DOC-21", "categoria": "Operativo", "actividad": "Resolución_Incidentes_Críticos", "tecnologia": "Contenedor",
         "texto": "Guía rápida de soporte para reiniciar servicios y recuperar aplicaciones web caídas."},
    ]
    return pd.DataFrame(datos)

# Motor de recomendación

In [4]:
class SistemaRecomendacionNexia:
    def __init__(self, df_docs: pd.DataFrame, usar_embeddings: bool = True):
        self.df = df_docs.reset_index(drop=True).copy()
        self.usar_embeddings = usar_embeddings and EMBEDDINGS_DISPONIBLES

        # Combinamos texto + categoría + tecnología para enriquecer la representación semántica
        corpus_enriquecido = (
            self.df["categoria"] + ": " + self.df["texto"] + " (" + self.df["tecnologia"] + ")"
        ).tolist()

        if self.usar_embeddings:
            # Modelo ligero multilingüe (soporta español e inglés técnico)
            self.modelo = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")
            self.vectores_docs = self.modelo.encode(corpus_enriquecido, normalize_embeddings=True)
            self.modo = "Embeddings Semánticos (Sentence-BERT)"
        else:
            stop_words_es = [
                "de", "la", "el", "los", "las", "y", "en", "para", "del", "al",
                "un", "una", "con", "por", "sobre", "entre", "o", "a", "que", "se", "su", "sus", "es"
            ]
            self.vectorizador = TfidfVectorizer(
                stop_words=stop_words_es,
                strip_accents="unicode",
                ngram_range=(1, 2)
            )
            self.vectores_docs = self.vectorizador.fit_transform(corpus_enriquecido).toarray()
            self.modo = "TF-IDF (N-gramas)"

        # Matriz precalculada de similitud documento-a-documento (N x N)
        self.matriz_similitud_docs = cosine_similarity(self.vectores_docs)

    def _vectorizar_textos(self, textos: list[str]) -> np.ndarray:
        if self.usar_embeddings:
            return self.modelo.encode(textos, normalize_embeddings=True)
        return self.vectorizador.transform(textos).toarray()

    def agregar_documento(self, texto: str, categoria: str, actividad: str = "General", tecnologia: str = "Base_Conocimiento"):
        """Permite conectar el clasificador automático para indexar documentos nuevos en caliente."""
        nuevo_id = f"DOC-{len(self.df) + 1:02d}"
        nueva_fila = pd.DataFrame([{
            "id": nuevo_id, "categoria": categoria, "actividad": actividad,
            "tecnologia": tecnologia, "texto": texto
        }])
        self.df = pd.concat([self.df, nueva_fila], ignore_index=True)
        self.__init__(self.df, usar_embeddings=self.usar_embeddings)

    # MECANISMO A: Recomendación por similitud semántica entre documentos
    def recomendar_por_documento(self, doc_id: str, top_k: int = 3, boost_ontologia: float = 0.05) -> pd.DataFrame:
        """
        Dado un documento seleccionado/abierto por el usuario, sugiere los top_k
        documentos más parecidos semánticamente, aplicando un ligero impulso
        si comparten la misma Actividad o Tecnología en la ontología.
        """
        if doc_id not in self.df["id"].values:
            raise ValueError(f"El ID '{doc_id}' no existe en la base de conocimiento.")

        idx = self.df.index[self.df["id"] == doc_id][0]
        doc_base = self.df.iloc[idx]

        scores = self.matriz_similitud_docs[idx].copy()

        # Refuerzo ontológico ligero (misma actividad o misma herramienta)
        misma_actividad = (self.df["actividad"] == doc_base["actividad"]).astype(float) * boost_ontologia
        misma_tecnologia = (self.df["tecnologia"] == doc_base["tecnologia"]).astype(float) * boost_ontologia
        scores_finales = scores + misma_actividad + misma_tecnologia

        resultados = self.df.copy()
        resultados["similitud_semantica"] = scores
        resultados["score_final"] = scores_finales

        # Excluir el mismo documento consultado
        resultados = resultados[resultados["id"] != doc_id]
        return resultados.sort_values("score_final", ascending=False).head(top_k)

    
    # MECANISMO B: Recomendación basada en historial de consultas y contexto
    def recomendar_por_contexto(
        self,
        historial_consultas: list[str],
        docs_vistos: list[str] | None = None,
        top_k: int = 3,
        umbral: float = 0.15
    ) -> pd.DataFrame:
        """
        Construye un vector de interés del usuario a partir de:
        1) Su historial de búsquedas (con decaimiento exponencial: las consultas
           más recientes tienen mayor peso).
        2) Los documentos ya abiertos en la sesión (filtrándolos para no repetir).
        """
        docs_vistos = docs_vistos or []
        vectores_contexto = []
        pesos = []

        # 1. Ponderar consultas recientes (más peso a las últimas consultas)
        if historial_consultas:
            vecs_q = self._vectorizar_textos(historial_consultas)
            n_q = len(historial_consultas)
            for i, vec in enumerate(vecs_q):
                peso_recencia = 0.6 ** (n_q - 1 - i)  # La más reciente vale 1.0, la anterior 0.6, etc.
                vectores_contexto.append(vec)
                pesos.append(peso_recencia)

        # 2. Incorporar documentos consultados previamente como señal implícita de interés
        for doc_id in docs_vistos:
            if doc_id in self.df["id"].values:
                idx = self.df.index[self.df["id"] == doc_id][0]
                vectores_contexto.append(self.vectores_docs[idx])
                pesos.append(0.5)

        if not vectores_contexto:
            return pd.DataFrame()

        # Vector perfil del usuario (promedio ponderado)
        perfil_usuario = np.average(vectores_contexto, axis=0, weights=pesos).reshape(1, -1)
        similitudes = cosine_similarity(perfil_usuario, self.vectores_docs).flatten()

        resultados = self.df.copy()
        resultados["afinidad_contexto"] = similitudes

        # Filtrar documentos ya vistos y los que no superan el umbral mínimo
        resultados = resultados[~resultados["id"].isin(docs_vistos)]
        resultados = resultados[resultados["afinidad_contexto"] >= umbral]

        return resultados.sort_values("afinidad_contexto", ascending=False).head(top_k)

# Pruebas y Validación

In [5]:
# Inicialización del motor
df_nexia = cargar_base_conocimiento()
recomendador = SistemaRecomendacionNexia(df_nexia, usar_embeddings=True)
print(f"Motor activo: {recomendador.modo}\n")

# --- PRUEBA 1: Similitud semántica entre documentos ---
doc_actual = "DOC-15"  # Reporte post-mortem del fallo en el clúster de producción de AWS
info_doc = df_nexia[df_nexia["id"] == doc_actual].iloc[0]
print(f" Documento abierto: [{info_doc['id']}] ({info_doc['categoria']}) {info_doc['texto']}")
print(" Documentos recomendados relacionados:")
recs_doc = recomendador.recomendar_por_documento(doc_actual, top_k=3)
for _, row in recs_doc.iterrows():
    print(f"   • [{row['id']}] [{row['categoria']}] (Score: {row['score_final']:.2f}) -> {row['texto']}")

print("\n" + "-" * 75 + "\n")

# --- PRUEBA 2: Recomendación basada en historial de consultas y sesión ---
historial_usuario = [
    "cómo estructurar tablas de clientes",
    "modelos predictivos y variables de entrada"
]
documentos_ya_leidos = ["DOC-10"]  # Ya leyó el Diccionario de datos de ML

print(f" Historial de búsqueda: {historial_usuario}")
print(f" Documentos ya consultados en sesión: {documentos_ya_leidos}")
print(" Sugerencias personalizadas según tu contexto:")
recs_ctx = recomendador.recomendar_por_contexto(
    historial_consultas=historial_usuario,
    docs_vistos=documentos_ya_leidos,
    top_k=3
)
for _, row in recs_ctx.iterrows():
    print(f"   • [{row['id']}] [{row['categoria']}] (Afinidad: {row['afinidad_contexto']:.2f}) -> {row['texto']}")

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 18051.92it/s]


Motor activo: Embeddings Semánticos (Sentence-BERT)

 Documento abierto: [DOC-15] (Operativo) Reporte post-mortem del fallo en el clúster de producción de AWS.
 Documentos recomendados relacionados:
   • [DOC-17] [Operativo] (Score: 0.69) -> Ticket de resolución de incidente crítico de latencia en la base de datos.
   • [DOC-12] [Técnico] (Score: 0.67) -> Diagramas de flujo de datos y arquitectura de microservicios en AWS.
   • [DOC-19] [Operativo] (Score: 0.63) -> Bitácora de mantenimiento semanal y rotación de credenciales en servidores de desarrollo.

---------------------------------------------------------------------------

 Historial de búsqueda: ['cómo estructurar tablas de clientes', 'modelos predictivos y variables de entrada']
 Documentos ya consultados en sesión: ['DOC-10']
 Sugerencias personalizadas según tu contexto:
   • [DOC-18] [Operativo] (Afinidad: 0.49) -> Manual de usuario paso a paso para el sistema interno de gestión de viáticos.
   • [DOC-14] [Técnico] (Afinida